# Predicting hiking moving time with linear regression

This notebook builds our first **supervised regression** models from real recorded
hikes. Supervised means the training rows contain the answer; regression means
the answer is a number. Our answer is **moving time in minutes**, not elapsed time.

By the end, you will be able to identify features and a target, fit ordinary least
squares with scikit-learn, interpret its parameters, measure held-out prediction
errors, and add segment predictions to estimate route moving time.

We compare three prespecified models:

| Model | Features known before walking the route |
| :--- | :--- |
| A: simple linear regression | Distance in miles |
| B: multiple linear regression | Distance, elevation gain, elevation loss |
| C: multiple linear regression with a category | Model B's features plus known hike type |

**Evidence comes from running the cells.** C is skipped when metadata or training
category coverage is insufficient. There is no fabricated training data or
invented performance. Synthetic fixtures appear only in the automated tests.

Run the GPX importer first, then run this notebook from top to bottom in the
project's `.venv` kernel. Reusable quality and evaluation functions live in
`src/hiking_time_predictor/modeling.py`; the main fitting steps are shown here too.

## 1. Load the real observations

A row is an approximately 0.5-mile distance segment, with a shorter final segment
when needed. Several rows belong to the same hike, so **rows are not independent
hikes**. The CSV's `hike_id` links related rows; it is never a predictor.

The processed measurements estimate accepted recorded movement. GPS gaps do not
reconstruct missing distance, ascent, or travel time. We preserve the source CSVs
and write only analysis outputs to `data/processed/regression/`.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

from hiking_time_predictor.modeling import (
    GEOMETRY, MODEL_FEATURES, MODEL_LABELS, NAISMITH_LABEL, TARGET,
    aggregate_hike_predictions, coefficient_table, error_metrics,
    evaluate_models, fit_model, group_splits, make_estimator,
    model_c_status, naismith_minutes, predict_route, prepare_segments,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
SEGMENTS_PATH = PROCESSED / 'hiking_segments.csv'
if not SEGMENTS_PATH.is_file():
    raise FileNotFoundError('Start in the project root or notebooks folder and run the GPX importer first.')
OUTPUT = PROCESSED / 'regression'
OUTPUT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 16)
pd.set_option('display.precision', 3)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

def show_figure(fig, filename):
    fig.savefig(OUTPUT / filename, dpi=160, bbox_inches='tight')
    plt.show()
    plt.close(fig)

raw_segments = pd.read_csv(SEGMENTS_PATH, dtype={'hike_id': 'string', 'hike_type': 'string'})
summary_path = PROCESSED / 'hiking_summary.csv'
summary = pd.read_csv(summary_path, dtype={'hike_id': 'string'}) if summary_path.exists() else None
versions = {name: importlib.metadata.version(name)
            for name in ['numpy', 'pandas', 'scikit-learn', 'matplotlib']}
print('Package versions:', versions)
print('Python kernel:', sys.executable)
print(f'{len(raw_segments)} segments from {raw_segments.hike_id.nunique()} recorded hikes')
display(raw_segments.head())

## 2. Decide which observations can support a model

Bad labels and incomplete geometry can teach a model the wrong relationship.
We apply a fixed, conservative policy before looking at prediction errors:

- Distance and moving time must be positive; gain and loss must be nonnegative.
  All must be finite numbers. Row identifiers must be valid and unique.
- Unknown time must be zero. Every measurement-quality flag excludes the row,
  including an unfamiliar new flag. The audit records each reason.
- `missing_hike_type` is allowed for A/B because neither uses that category. C
  requires a supplied `day_hike` or `backpacking` label. We do not guess metadata.
- Ordinary stationary time is allowed: the target already excludes stops. A
  `user_confirmed_rest` flag still requires review under this conservative policy.
- A shorter final segment is valid and stays in the dataset. We do not discard
  difficult predictions after seeing the results.

This is separate from the older `select_model_ready_segments` EDA helper, which
requires known hike type for every selected row. Keeping A/B's metadata rule
separate lets us learn from sound geometry without manufacturing categories.

In [ ]:
audit = prepare_segments(raw_segments)
eligible = audit.loc[audit.eligible_ab].copy().reset_index(drop=True)
known_type = audit.loc[audit.eligible_c].copy().reset_index(drop=True)
quality_by_hike = audit.groupby('hike_id', dropna=False).agg(
    exported_segments=('segment_index', 'size'),
    eligible_ab=('eligible_ab', 'sum'), eligible_c=('eligible_c', 'sum'),
)
quality_by_hike['excluded_ab'] = quality_by_hike.exported_segments - quality_by_hike.eligible_ab
display(quality_by_hike)
print(f'A/B: {len(eligible)} eligible segments from {eligible.hike_id.nunique()} hikes')
print(f'C: {len(known_type)} eligible segments with known hike type')
display(audit.loc[~audit.eligible_ab, 'exclusion_reasons_ab'].value_counts().rename('rows'))
audit.to_csv(OUTPUT / 'segment_quality_audit.csv', index=False)
quality_by_hike.to_csv(OUTPUT / 'quality_by_hike.csv')
if eligible.empty:
    print('No model-ready segments: review source quality before training. No scores will be created.')

## 3. Separate features X from target y, and split by hike

**X** is the table of input features. Each column is information we could obtain
from a planned route. **y** is the observed answer, `moving_time_min`. Fitting
uses both X and y; predicting a future route uses X alone.

We explicitly allow only the requested predictors. `hike_id` and `segment_index`
organize evaluation. Actual speed, pace, timestamps, elapsed/observed time,
stationary/unknown time, and any value computed from the answer never enter X.
Quality fields help choose rows; they are not features.

Neighboring segments share a hiker, terrain, weather, and recording conditions.
A random row split would allow parts of the same outing into both sets. Instead:

- With 2–20 eligible hikes, **LeaveOneGroupOut** trains on all but one hike and
  predicts that held-out hike. Each eligible row receives one held-out prediction.
- With more than 20 hikes, one **GroupShuffleSplit** reserves 25% of hike IDs with
  a fixed seed. This is a practical teaching threshold, not a guarantee of enough data.
- With zero or one eligible hike, between-hike evaluation is unavailable. Fitting
  can demonstrate the mechanics, but cannot establish future-hike generalization.

The same rows and folds are used for all models within a comparison. No parameters
are selected by trying many settings against these held-out results.

In [ ]:
X = eligible[GEOMETRY]
y = eligible[TARGET]
groups = eligible['hike_id']  # Used only by the splitter.
splits, split_method = group_splits(eligible)
print('X shape:', X.shape, '| y shape:', y.shape)
print('Evaluation:', split_method, '| folds:', len(splits))
if splits:
    train_index, test_index = splits[0]
    train, test = eligible.iloc[train_index], eligible.iloc[test_index]
    assert set(train.hike_id).isdisjoint(set(test.hike_id))
    print('First fold:', train.hike_id.nunique(), 'training hikes;', test.hike_id.nunique(), 'test hike(s)')
    print('Held-out IDs:', test.hike_id.unique().tolist())
    eda_data = train
else:
    eda_data = eligible
    print('Any EDA/fits below are descriptive only; there is no between-hike test set.')

## 4. Look for relationships without changing the test

We inspect only the first fold's training rows here. All feature sets and quality
rules were fixed above. The heatmap shows **Pearson correlation**, which measures
linear association from −1 to +1. Correlation does not prove causation. The three
scatter plots show moving time against each geometry feature; colors identify
hikes to make clustering visible.

Most distances are deliberately about 0.5 miles. Consequently the distance slope
can depend strongly on the few shorter final segments. Gain and loss can also
be correlated, making separate coefficients unstable. A missing heatmap value
means there was insufficient variation to calculate a correlation.

In [ ]:
if not eda_data.empty:
    display(eda_data[[*GEOMETRY, TARGET]].describe())
    print('Full half-mile rows:', np.isclose(eda_data.distance_miles, 0.5).sum(), '/', len(eda_data))
    correlations = eda_data[[*GEOMETRY, TARGET]].corr()
    display(correlations)
    labels = ['Distance (mi)', 'Gain (ft)', 'Loss (ft)', 'Moving time (min)']
    fig, ax = plt.subplots(figsize=(7, 5))
    heatmap = ax.imshow(correlations, cmap='coolwarm', vmin=-1, vmax=1)
    ax.set_xticks(range(4), labels, rotation=30, ha='right')
    ax.set_yticks(range(4), labels)
    for row in range(4):
        for column in range(4):
            value = correlations.iloc[row, column]
            ax.text(column, row, f'{value:.2f}' if np.isfinite(value) else 'n/a', ha='center', va='center')
    fig.colorbar(heatmap, ax=ax, label='Pearson correlation')
    ax.set_title('Relationships in the first training fold')
    show_figure(fig, 'training_correlations.png')

    fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True, layout='constrained')
    for hike_number, (_, hike) in enumerate(eda_data.groupby('hike_id')):
        for ax, feature, label in zip(axes, GEOMETRY, labels[:3]):
            ax.scatter(hike[feature], hike[TARGET], color=plt.get_cmap('tab10')(hike_number % 10), alpha=0.8)
            ax.set_xlabel(label)
    axes[0].set_ylabel('Moving time (min)')
    fig.suptitle('Segment geometry and moving time — colors distinguish training hikes')
    show_figure(fig, 'training_relationships.png')

## 5. Fit ordinary least squares, one fold at a time

Linear regression predicts a weighted sum of the features:

\[
\hat y_A = b_0 + b_d d,
\qquad
\hat y_B = b_0 + b_d d + b_g g + b_l l.
\]

`LinearRegression.fit(X_train, y_train)` chooses the intercept and coefficients
that minimize the sum of squared training residuals. `predict(X_test)` then uses
those learned numbers without seeing the held-out answers. Lower training error
alone does not demonstrate better future predictions.

The following is an explicit worked example for the first fold. The reusable
evaluation loop later repeats the same procedure for every held-out hike.

In [ ]:
worked_models = {}
if splits:
    X_train, X_test = train[GEOMETRY], test[GEOMETRY]
    y_train, y_test = train[TARGET], test[TARGET]
    model_a = LinearRegression()
    model_a.fit(X_train[['distance_miles']], y_train)
    prediction_a = model_a.predict(X_test[['distance_miles']])
    model_b = LinearRegression()
    model_b.fit(X_train[GEOMETRY], y_train)
    prediction_b = model_b.predict(X_test[GEOMETRY])
    worked_models = {'A': model_a, 'B': model_b}
    display(pd.DataFrame({'actual_moving_min': y_test.to_numpy(),
                          'A_predicted_min': prediction_a, 'B_predicted_min': prediction_b}))
else:
    print('Worked train/test example skipped: need at least two eligible hikes.')

### Interpret the learned numbers

The intercept $b_0$ is predicted minutes when all numeric features are zero
(and, for C, the reference category applies). That point is outside our normal
segment data. It is a mathematical offset, not a measured rest or setup time.

For B, holding the other features fixed:

- $b_d$ is additional minutes per mile.
- $100b_g$ is additional minutes per 100 feet of ascent.
- $100b_l$ is additional minutes per 100 feet of descent. Loss is recorded as a
  positive magnitude. A positive loss coefficient predicts slower descent.

Coefficients describe associations in these data. A sign can change with small
samples or correlated inputs; it is not proof of an effect. `design_rank` below
is the rank of the centered feature matrix. A rank below `feature_count` signals
that separate slopes cannot be uniquely identified. Full rank does not guarantee
stable or causal coefficients. A and B keep original units, so larger raw
coefficient magnitude does not automatically mean greater feature importance.

In [ ]:
if worked_models:
    worked_coefficients = pd.concat([
        coefficient_table(model, name).assign(model=MODEL_LABELS[name])
        for name, model in worked_models.items()
    ], ignore_index=True)
    display(worked_coefficients)
    b = worked_models['B']
    print(f'First-fold B: {b.intercept_:.3f} + {b.coef_[0]:.3f} × miles '
          f'+ {b.coef_[1]:.5f} × gain_ft + {b.coef_[2]:.5f} × loss_ft')
    print(f'100 extra ascent feet: {100*b.coef_[1]:.2f} predicted minutes; '
          f'100 extra descent feet: {100*b.coef_[2]:.2f} predicted minutes.')

### Model C: encode known categories inside the training fold

A string cannot be used as a numeric slope. C uses `OneHotEncoder(drop='first')`
inside a `Pipeline`, followed by `LinearRegression`. With both categories present,
one category is the reference and the other is represented by a 0/1 column. Its
coefficient is the predicted per-segment difference from the reference category,
holding geometry fixed. The fitted encoder below reports the reference explicitly.

C needs both categories in **every training fold**. Otherwise a held-out category's
effect has never been learned. Missing or unrecognized types are never silently
assigned to the reference. If C is available, we rerun **A, B, C and Naismith on
the same known-metadata cohort and grouped folds**. Those scores form a separate
comparison from A/B on the larger geometry cohort.

In [ ]:
c_ready, c_message = model_c_status(known_type)
print(c_message)
if c_ready:
    c_splits, _ = group_splits(known_type)
    c_train_index, _ = c_splits[0]
    c_train = known_type.iloc[c_train_index]
    model_c = make_estimator('C')
    model_c.fit(c_train[MODEL_FEATURES['C']], c_train[TARGET])
    encoder = model_c.named_steps['features'].named_transformers_['type']
    print('Reference category:', encoder.categories_[0][0])
    display(coefficient_table(model_c, 'C'))

## 6. Define a comparable Naismith baseline

We use the **original imperial variant** explicitly: one hour per 3 horizontal
miles plus one hour per 2,000 ascent feet. Thus

\[
\hat t_{N,\mathrm{min}} = 20d_{\mathrm{miles}} + 0.03g_{\mathrm{ft}}.
\]

This version adds no breaks or downhill adjustment. It has fixed coefficients and
requires no fitting. The closely related 5-km/600-m version has slightly different
constants; we do not mix the two. The imperial definition is documented in the
[Yorkshire Ramblers' Club Journal, volume 11, no. 39](https://www.yrc.org.uk/wp-content/uploads/2020/05/YRCJ-11-39-1985-OCR.pdf).

For a fair comparison, every method predicts **moving minutes on the same retained
distance and smoothed ascent**. We compare to `moving_time_min`, never elapsed
time including breaks or unknown gaps. Naismith is a walking-time heuristic;
the GPX movement detector's definition of movement remains imperfect. We do not
calibrate the baseline using test answers.

In [ ]:
if not eligible.empty:
    baseline_example = eligible[[*GEOMETRY, TARGET]].head().copy()
    baseline_example['naismith_min'] = naismith_minutes(baseline_example)
    display(baseline_example)

## 7. Evaluate only held-out predictions

For $n$ predictions, our two scores are

\[
\mathrm{MAE}=\frac{1}{n}\sum_i |y_i-\hat y_i|,\qquad
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_i(y_i-\hat y_i)^2}.
\]

Both are in minutes and lower is better. MAE is average absolute error; RMSE
penalizes a few large errors more strongly. Bias here is **predicted minus actual**:
positive means overprediction. We retain any negative OLS outputs in evaluation
and count them, so an impossible prediction cannot be hidden by clipping it.

Pooled segment metrics give every eligible segment equal weight, so longer retained
hikes contribute more rows. The per-hike table and its mean MAE give each hike
equal weight. With so few hikes, neither score is a precise population estimate.

In [ ]:
evaluation = evaluate_models(eligible, ('A', 'B'))
predictions = evaluation.predictions
segment_metrics = error_metrics(predictions)
print(evaluation.method)
display(evaluation.folds)
display(segment_metrics)

per_hike_frames = []
for hike_id, hike_predictions in predictions.groupby('hike_id'):
    per_hike_frames.append(error_metrics(hike_predictions).assign(hike_id=hike_id))
per_hike_metrics = pd.concat(per_hike_frames, ignore_index=True) if per_hike_frames else pd.DataFrame()
if not per_hike_metrics.empty:
    print('Segment MAE within each held-out hike (minutes):')
    display(per_hike_metrics.pivot(index='hike_id', columns='model', values='MAE_min'))
    print('Mean of per-hike segment MAEs (each hike has equal weight):')
    display(per_hike_metrics.groupby('model').MAE_min.mean().rename('mean_hike_segment_MAE_min'))

c_evaluation = evaluate_models(known_type, ('A', 'B', 'C')) if c_ready else None
if c_evaluation is not None:
    print('Separate fair comparison on known-metadata rows only:')
    display(error_metrics(c_evaluation.predictions))

## 8. Diagnose actual-versus-predicted values and residuals

In the first plot, perfect predictions lie on the diagonal. Points above it are
overpredictions; below it are underpredictions. The horizontal and vertical axes
use the same limits, so distances from the diagonal are comparable across models.

A **residual** is actual minus predicted, the opposite sign of the bias reported
above. A positive residual means the model underestimated time. Residuals should
ideally scatter around zero. Curvature suggests a missed nonlinear relationship;
a widening spread suggests changing error variance. A small sample cannot establish
that these assumptions hold. Every dot below is held out from its fitted model.

In [ ]:
def diagnostic_plots(frame, suffix):
    if frame.empty:
        return
    names = list(frame.model.unique())
    palette = ['#2463a6', '#d36a21', '#477a4a', '#9256a0']
    all_values = frame[[TARGET, 'predicted_time_min']].to_numpy()
    lower = min(0, float(all_values.min()))
    upper = float(all_values.max()) * 1.08 + 0.1
    fig, axes = plt.subplots(1, len(names), figsize=(4.2*len(names), 4), squeeze=False, layout='constrained')
    for ax, name, color in zip(axes[0], names, palette):
        points = frame.loc[frame.model.eq(name)]
        ax.scatter(points[TARGET], points.predicted_time_min, color=color, alpha=0.8)
        ax.plot([lower, upper], [lower, upper], color='0.4', linestyle='--')
        ax.set(xlim=(lower, upper), ylim=(lower, upper), xlabel='Actual moving time (min)',
               ylabel='Predicted moving time (min)', title=name)
        ax.set_aspect('equal', adjustable='box')
    fig.suptitle('Held-out segment predictions')
    show_figure(fig, f'actual_vs_predicted_{suffix}.png')

    residuals = frame[TARGET] - frame.predicted_time_min
    bound = max(float(residuals.abs().max()) * 1.12, 0.5)
    fig, axes = plt.subplots(1, len(names), figsize=(4.2*len(names), 4), squeeze=False, layout='constrained')
    for ax, name, color in zip(axes[0], names, palette):
        points = frame.loc[frame.model.eq(name)]
        ax.scatter(points.predicted_time_min, points[TARGET] - points.predicted_time_min, color=color, alpha=0.8)
        ax.axhline(0, color='0.4', linestyle='--')
        ax.set(xlabel='Predicted moving time (min)', ylabel='Actual − predicted (min)',
               ylim=(-bound, bound), title=name)
    fig.suptitle('Held-out segment residuals')
    show_figure(fig, f'residuals_{suffix}.png')

diagnostic_plots(predictions, 'geometry_cohort')
if c_evaluation is not None:
    diagnostic_plots(c_evaluation.predictions, 'known_type_cohort')

## 9. Add segment predictions and evaluate hike totals carefully

A route's predicted moving time is the sum of its segment predictions. We add
**only held-out predictions for that hike** and compare with the sum of observed
moving minutes over exactly those same segments. Summing before calculating error
lets over- and underpredictions cancel, as they can for a real route total.

Filtering may remove most of a recording. A sum over the remaining segments is
then a **retained-portion total**, not a whole-hike estimate. The coverage table
shows segment counts and retained accepted distance fraction. We label a total
as a whole recorded hike only if every exported segment is included, the hike
summary has no measurement flags or unknown time, and its distance/time totals
match. If the summary is unavailable, full-recording quality cannot be verified.
Even full coverage refers to accepted recorded movement, not reconstructed travel
over unrecorded sections.

We report retained-portion totals separately from verified whole recorded hikes.
A whole-hike metric based on one hike is simply that outing's absolute error;
it cannot establish reliable route-level generalization.

In [ ]:
hike_totals = aggregate_hike_predictions(predictions, audit, summary)
whole_totals = hike_totals.loc[hike_totals.whole_hike_eligible].copy()
portion_totals = hike_totals.loc[~hike_totals.whole_hike_eligible].copy()
whole_metrics = error_metrics(whole_totals)
portion_metrics = error_metrics(portion_totals)
print('Coverage of evaluated hikes:')
display(hike_totals.drop_duplicates(['fold', 'hike_id'])[
    ['hike_id', 'retained_segments', 'exported_segments', 'distance_coverage_fraction', 'whole_hike_eligible']])
print('Retained portions only — do not interpret as whole-hike performance:')
display(portion_metrics)
print('Verified whole recorded hikes (accepted moving time):')
display(whole_totals[['model', 'hike_id', TARGET, 'predicted_time_min']])
display(whole_metrics)
if whole_totals.hike_id.nunique() < 2:
    print('Fewer than two whole hikes: no reliable claim about future whole-route performance.')

c_hike_totals = None
if c_evaluation is not None:
    c_hike_totals = aggregate_hike_predictions(c_evaluation.predictions, audit, summary)
    print('Known-type cohort: verified whole-hike comparison on the same rows for A/B/C/Naismith:')
    display(error_metrics(c_hike_totals.loc[c_hike_totals.whole_hike_eligible]))
    print('Known-type cohort: retained-portion comparison:')
    display(error_metrics(c_hike_totals.loc[~c_hike_totals.whole_hike_eligible]))

## 10. Compare models visually

The bars compare MAE and RMSE on matched observations. The whole-hike panel uses
a different scale and sample size from the segment panel. The route total plot
shows why a method that wins on average segment error need not win on a particular
whole hike. There are no confidence intervals: segments within a hike are related,
and the small number of independent hikes does not support precise claims.

In [ ]:
def metric_comparison(segment_scores, whole_scores, suffix):
    if segment_scores.empty:
        print('No held-out scores to plot.')
        return
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), layout='constrained')
    for ax, scores, title in [
        (axes[0], segment_scores, 'Held-out segments'),
        (axes[1], whole_scores, 'Whole recorded hikes'),
    ]:
        if scores.empty:
            ax.text(0.5, 0.5, 'No eligible whole hikes', ha='center', transform=ax.transAxes)
            ax.set_axis_off()
            continue
        positions = np.arange(len(scores))
        for offset, metric, color in [(-0.18, 'MAE_min', '#2463a6'), (0.18, 'RMSE_min', '#d36a21')]:
            bars = ax.bar(positions + offset, scores[metric], width=0.36, label=metric.replace('_min', ''), color=color)
            ax.bar_label(bars, fmt='%.2f', padding=3, fontsize=9)
        ax.set_xticks(positions, [name.split(':')[0] if ':' in name else 'Naismith' for name in scores.model])
        ax.set_ylabel('Error (minutes)')
        ax.set_title(f'{title}: n={int(scores.observations.iloc[0])}, {int(scores.hikes.iloc[0])} hike(s)')
        ax.margins(y=0.22)
        ax.legend()
    fig.suptitle('Matched comparison — lower error is better')
    show_figure(fig, f'model_comparison_{suffix}.png')

metric_comparison(segment_metrics, whole_metrics, 'geometry_cohort')
if c_evaluation is not None:
    metric_comparison(error_metrics(c_evaluation.predictions),
                      error_metrics(c_hike_totals.loc[c_hike_totals.whole_hike_eligible]), 'known_type_cohort')

if not whole_totals.empty:
    routes = list(whole_totals.hike_id.unique())
    fig, ax = plt.subplots(figsize=(9, 4.5), layout='constrained')
    positions = np.arange(len(routes))
    names = list(whole_totals.model.unique())
    width = 0.8 / (len(names) + 1)
    actual = whole_totals.drop_duplicates('hike_id').set_index('hike_id').loc[routes, TARGET]
    bars = ax.bar(positions - 0.4, actual, width=width, align='edge', color='0.3', label='Actual moving time')
    ax.bar_label(bars, fmt='%.1f', padding=3)
    for number, name in enumerate(names, start=1):
        values = whole_totals.loc[whole_totals.model.eq(name)].set_index('hike_id').loc[routes, 'predicted_time_min']
        bars = ax.bar(positions - 0.4 + number*width, values, width=width, align='edge', label=name)
        ax.bar_label(bars, fmt='%.1f', padding=3)
    ax.set_xticks(positions, [f'Recorded hike {number+1}' for number in range(len(routes))])
    ax.set(ylabel='Moving time (minutes)', title='Whole recorded hikes: sum of held-out segment predictions')
    ax.margins(y=0.3)
    ax.legend(fontsize=8)
    show_figure(fig, 'whole_hike_predictions.png')

## 11. Predict an entire future route

After evaluation, we may refit each available model on all its eligible observations
for later use. Scores reported above still come from the earlier held-out fits.
An all-data fit's training predictions are not new validation evidence.

A planned route needs **spatial geometry**, with approximately 0.5-mile segments
and its shorter final segment, the same units, and compatible elevation smoothing.
It does not need timestamps, observed time, speed, or answers. C additionally
needs one known hike type for the entire route. The existing recorded-GPX importer
is not a route planner: do not invent timestamps to process an untimed route.

For $N$ segments, adding predictions from B gives

\[
\hat T = \sum_{i=1}^N\hat t_i
       = Nb_0 + b_d\sum_i d_i + b_g\sum_i g_i + b_l\sum_i l_i.
\]

Calling a segment model once with whole-route totals would include the intercept
only once; changing segment size changes how often the offset is added. C's
category offset also repeats per segment. Keep the segmentation convention fixed
and inspect very short tails. We reject negative future predictions rather than
silently replacing them with zero. Small-sample route errors and extrapolation
still need scrutiny; summing segment RMSEs is not a valid uncertainty interval.
Breaks are a separate planning input and are not learned from these moving labels.

In [ ]:
final_models = {name: fit_model(eligible, name) for name in ('A', 'B')} if not eligible.empty else {}
if c_ready:
    final_models['C'] = fit_model(known_type, 'C')
final_coefficients = pd.concat([
    coefficient_table(model, name).assign(model=MODEL_LABELS[name])
    for name, model in final_models.items()
], ignore_index=True) if final_models else pd.DataFrame()
print('All-data parameters for later prediction (not held-out performance):')
display(final_coefficients)

# Historical replay demonstrates future-route arithmetic using only its geometry.
# Exclude the replay hike entirely from training, even though it is a real recording.
replay_ids = whole_totals.hike_id.unique()
if len(replay_ids) and eligible.hike_id.nunique() >= 2:
    replay_id = replay_ids[0]
    replay_train = eligible.loc[~eligible.hike_id.eq(replay_id)]
    route_geometry = audit.loc[audit.hike_id.eq(replay_id), GEOMETRY].reset_index(drop=True)
    replay_estimator = fit_model(replay_train, 'B')
    route_predictions = predict_route(replay_estimator, 'B', route_geometry)
    predicted_total_min = route_predictions.predicted_time_min.sum()
    display(route_predictions)
    print('Historical replay, not a newly supplied future route:', replay_id)
    print(f'B route moving time: {predicted_total_min:.2f} min ({predicted_total_min/60:.2f} hours)')
    print(f'Naismith on identical route geometry: {naismith_minutes(route_geometry).sum():.2f} min')
    route_predictions.to_csv(OUTPUT / 'historical_route_replay.csv', index=False)
else:
    print('No verified complete held-out recording is available for a whole-route replay.')

# Optional actual future route: supply geometry from a route-planning source.
# CSV headers: distance_miles,elevation_gain_ft,elevation_loss_ft[,hike_type]
future_route_path = PROJECT_ROOT / 'data' / 'future' / 'route_segments.csv'
if future_route_path.exists() and final_models:
    future_route = pd.read_csv(future_route_path)
    for name, model in final_models.items():
        if name == 'C' and ('hike_type' not in future_route
                            or not future_route.hike_type.isin(['day_hike', 'backpacking']).all()
                            or future_route.hike_type.nunique() != 1):
            print('Future C prediction skipped: supply one known hike_type for the route.')
            continue
        route_result = predict_route(model, name, future_route)
        print(f'{MODEL_LABELS[name]} future route: {route_result.predicted_time_min.sum():.2f} moving minutes')
        route_result.to_csv(OUTPUT / f'future_route_{name}.csv', index=False)
    print(f'Naismith future route: {naismith_minutes(future_route).sum():.2f} moving minutes')
    print('Compare future geometry to training ranges before trusting extrapolations:')
    display(pd.DataFrame({'training_min': eligible[GEOMETRY].min(), 'training_max': eligible[GEOMETRY].max(),
                          'future_min': future_route[GEOMETRY].min(), 'future_max': future_route[GEOMETRY].max()}))
else:
    print('No future route supplied. To use this workflow, create data/future/route_segments.csv with spatial geometry.')

## 12. Save actual findings and interpret their limits

The output tables contain the quality decisions, hike split assignments, every
held-out prediction, fold coefficients, and metrics. File hashes identify the
input snapshots. Rerunning after importing additional real hikes recomputes all
findings; it does not append test fixtures to the observations.

Read the results at three levels: segment errors on the clean cohort, errors in
the retained portions of incomplete recordings, and the few verified whole hikes.
They answer different questions and should not be combined into a single claim.

In [ ]:
exports = {
    'held_out_segment_predictions.csv': predictions,
    'grouped_folds.csv': evaluation.folds,
    'fold_coefficients.csv': evaluation.coefficients,
    'segment_metrics.csv': segment_metrics,
    'per_hike_segment_metrics.csv': per_hike_metrics,
    'hike_totals_and_coverage.csv': hike_totals,
    'retained_portion_metrics.csv': portion_metrics,
    'whole_hike_metrics.csv': whole_metrics,
    'final_fit_coefficients.csv': final_coefficients,
}
if c_evaluation is not None:
    exports.update({
        'known_type_segment_predictions.csv': c_evaluation.predictions,
        'known_type_folds.csv': c_evaluation.folds,
        'known_type_fold_coefficients.csv': c_evaluation.coefficients,
        'known_type_segment_metrics.csv': error_metrics(c_evaluation.predictions),
        'known_type_hike_totals.csv': c_hike_totals,
        'known_type_whole_hike_metrics.csv': error_metrics(c_hike_totals.loc[c_hike_totals.whole_hike_eligible]),
        'known_type_retained_portion_metrics.csv': error_metrics(c_hike_totals.loc[~c_hike_totals.whole_hike_eligible]),
    })
for filename, table in exports.items():
    table.to_csv(OUTPUT / filename, index=False)

run_summary = {
    'input_sha256': hashlib.sha256(SEGMENTS_PATH.read_bytes()).hexdigest(),
    'summary_sha256': hashlib.sha256(summary_path.read_bytes()).hexdigest() if summary_path.exists() else None,
    'package_versions': versions,
    'exported_segments': len(audit), 'exported_hikes': int(audit.hike_id.nunique()),
    'eligible_segments_ab': len(eligible), 'eligible_hikes_ab': int(eligible.hike_id.nunique()),
    'excluded_segments_ab': int((~audit.eligible_ab).sum()),
    'evaluation_method': evaluation.method, 'model_c_status': c_message,
    'verified_whole_hikes': int(whole_totals.hike_id.nunique()),
    'naismith_definition': '20 min/mile + 0.03 min/ascent_ft; no breaks or descent adjustment',
}
(OUTPUT / 'run_summary.json').write_text(json.dumps(run_summary, indent=2) + '\n')

findings = [
    f'### Findings from this execution',
    f'- Source: **{len(audit)} segments / {audit.hike_id.nunique()} hikes**.',
    f'- Retained for A/B: **{len(eligible)} segments / {eligible.hike_id.nunique()} hikes**; '
    f'**{int((~audit.eligible_ab).sum())} segments excluded** by the measurement-quality policy.',
    f'- Evaluation: **{evaluation.method}**, with {len(evaluation.folds)} grouped folds.',
    f'- {c_message}',
]
for row in segment_metrics.itertuples():
    findings.append(f'- {row.model}: held-out segment MAE **{row.MAE_min:.2f} min**, '
                    f'RMSE **{row.RMSE_min:.2f} min** ({row.observations} segments, {row.hikes} hikes).')
findings.append(f'- Only **{whole_totals.hike_id.nunique()}** recorded hike(s) meet the whole-hike coverage/quality check.')
for row in whole_totals.itertuples():
    findings.append(f'- Whole-hike {row.model}: predicted **{row.predicted_time_min:.2f} min** '
                    f'versus **{row.moving_time_min:.2f} min** observed accepted movement.')
if not segment_metrics.empty:
    best_segment = segment_metrics.sort_values('MAE_min').iloc[0].model
    findings.append(f'- Lowest segment MAE in this run: **{best_segment}**. This is an exploratory comparison, not a deployment claim.')
if not whole_metrics.empty:
    best_whole = whole_metrics.sort_values('MAE_min').iloc[0].model
    findings.append(f'- Lowest whole-hike MAE in this run: **{best_whole}**, based on just {whole_totals.hike_id.nunique()} hike(s).')
findings_text = '\n'.join(findings)
display(Markdown(findings_text))
(OUTPUT / 'findings.md').write_text(findings_text + '\n')
print('Saved analysis outputs to:', OUTPUT)

### What these results can and cannot establish

1. **Few independent outings.** Many correlated segments do not replace many
   hikes. Even multiple outings can share the same hiker or trail; grouping by
   hike does not test new people or entirely new regions. A single retained hike
   never supports a generalization claim. More whole hikes and a later untouched
   test set are needed before selecting a model for dependable route planning.
2. **Selected data.** Conservative filtering removes flagged segments, so measured
   errors describe the retained clean sections. They may not apply to gaps, difficult
   GPS environments, or slow/technical terrain that the detector misclassifies.
3. **Estimated labels and features.** Moving time depends on GPX thresholds;
   elevation depends on smoothing. Both can be wrong. Future map geometry may
   differ from historical accepted-track geometry. No model restores missing travel.
4. **Restricted distance variation.** Almost all segments are half a mile. Distance
   and intercept estimates can hinge on a few short tails. A model can fit these
   rows while extrapolating poorly to different segment sizes or routes.
5. **Missing context.** Pack weight, trail surface, weather, fatigue, and fitness
   are not modeled. Unknown hike types stay unknown. A category effect learned
   from too few independent trips would be especially fragile.
6. **Exploratory validation.** Looking at these cross-validation results to prefer
   one model makes them development evidence. We have not performed a final,
   untouched prospective test or estimated a reliable prediction interval.

**Class discussion:** Why can Naismith beat a fitted model on a whole hike while
losing on segment MAE? What happens when the residuals on one hike have the same
sign? Why does doubling the number of segments change an intercept model's sum?

For implementation references, see scikit-learn's
[LinearRegression documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
and [grouped cross-validation guide](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data).